# Maize Assistant Asking/Greetings Model Training

This notebook trains a text intent model from the two CSV files and saves artifacts for `app.py`.

**Outputs**
- `askingmodelmaize.joblib`
- `askingmodelmaize_response_map.json`
- `askingmodelmaize_labels.json`

## 1. Load and Inspect CSV Datasets

In [24]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(r"d:\\CAPSTONE PROJECT\\datasets\\maize trainningsystem")
CSV_1 = DATA_DIR / "maize_assistant_training.csv"
CSV_2 = DATA_DIR / "maize_assistant_dataset (2).csv"


def load_intent_csv(path: Path) -> pd.DataFrame:
    """Parse intent CSVs even when responses contain unquoted commas."""
    rows = []
    with path.open("r", encoding="utf-8-sig") as f:
        header = f.readline().strip().split(",")
        if len(header) < 3:
            raise ValueError(f"Unexpected CSV header in {path}: {header}")

        for line_no, raw_line in enumerate(f, start=2):
            line = raw_line.strip()
            if not line:
                continue
            parts = line.split(",", 2)
            if len(parts) != 3:
                print(f"Skipping malformed row {line_no} in {path.name}")
                continue
            text, intent, response = (part.strip() for part in parts)
            text = text.strip('"')
            intent = intent.strip('"')
            response = response.strip().strip('"')
            if text and intent and response:
                rows.append({"text": text, "intent": intent, "response": response})

    if not rows:
        raise ValueError(f"No usable rows found in {path}")

    return pd.DataFrame(rows)


for path in (CSV_1, CSV_2):
    if not path.exists():
        raise FileNotFoundError(f"Missing CSV: {path}")

df_1 = load_intent_csv(CSV_1)
df_2 = load_intent_csv(CSV_2)

print("CSV 1 rows:", len(df_1))
print("CSV 2 rows:", len(df_2))
print("CSV 1 columns:", df_1.columns.tolist())
print("CSV 2 columns:", df_2.columns.tolist())
print("\nSample rows:")
print(df_1.head(5).to_string(index=False))

CSV 1 rows: 51
CSV 2 rows: 60
CSV 1 columns: ['text', 'intent', 'response']
CSV 2 columns: ['text', 'intent', 'response']

Sample rows:
          text   intent                                                                                                    response
         hello greeting Hello! I am Maize Assistant, developed by YUMVAGUSENGA PATRICK. How can I help with your maize crops today?
            hi greeting                              Hi! I am Maize Assistant. Ask me anything about maize diseases or crop health.
           hey greeting                                          Hey there! Maize Assistant is ready to help you. What do you need?
  good morning greeting                             Good morning! I am Maize Assistant. Let us keep your maize crops healthy today.
good afternoon greeting                  Good afternoon! I am here to help you detect and monitor maize diseases. How can I assist?


## 2. Normalize and Label Text Data

In [25]:
import re

INTENT_CANONICAL = {
    "greetings": "greeting",
    "hello": "greeting",
    "hi": "greeting",
    "ask": "asking",
    "asking": "asking",
    "question": "asking",
}


def clean_text(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_intent(intent: str) -> str:
    base = str(intent).strip().lower()
    return INTENT_CANONICAL.get(base, base)


# Combine and clean (use all rows from both CSV files)
_df = pd.concat([df_1, df_2], ignore_index=True)
_df["text_clean"] = _df["text"].apply(clean_text)
_df["intent_clean"] = _df["intent"].apply(normalize_intent)
_df["response_clean"] = _df["response"].astype(str).str.strip()

_df = _df[
    (_df["text_clean"] != "")
    & (_df["intent_clean"] != "")
    & (_df["response_clean"] != "")
].copy()

# Keep duplicates (user requested all CSV rows for training)
dup_count = _df.duplicated(subset=["text_clean", "intent_clean", "response_clean"]).sum()

df_combined = _df.reset_index(drop=True)

print("=" * 80)
print("CLEANED DATASET SUMMARY")
print("=" * 80)
print(f"Total rows (all CSV rows after cleaning): {len(df_combined)}")
print(f"Duplicate rows kept for training: {dup_count}")
print(f"Unique intents: {df_combined['intent_clean'].nunique()}")
print(f"Intents: {sorted(df_combined['intent_clean'].unique().tolist())}")
print(df_combined.head(8).to_string(index=False))
print("=" * 80)

CLEANED DATASET SUMMARY
Total rows (all CSV rows after cleaning): 111
Duplicate rows kept for training: 0
Unique intents: 18
Intents: ['advice', 'agriculture', 'capabilities', 'command', 'date_question', 'disease_info', 'education', 'goodbye', 'gratitude', 'greeting', 'help', 'identity', 'maize_question', 'maize_symptom', 'monitoring', 'progression', 'smalltalk', 'uncertainty']
          text   intent                                                                                                    response     text_clean intent_clean                                                                                              response_clean
         hello greeting Hello! I am Maize Assistant, developed by YUMVAGUSENGA PATRICK. How can I help with your maize crops today?          hello     greeting Hello! I am Maize Assistant, developed by YUMVAGUSENGA PATRICK. How can I help with your maize crops today?
            hi greeting                              Hi! I am Maize Assistant. Ask 

## 3. Split Data into Train and Validation Sets

In [26]:
from collections import Counter
from sklearn.model_selection import train_test_split

X = df_combined["text_clean"].tolist()
y = df_combined["intent_clean"].tolist()

label_counts = Counter(y)
print("Intent sample counts:")
for label, count in sorted(label_counts.items()):
    print(f"  {label}: {count}")

min_count = min(label_counts.values()) if label_counts else 0
can_stratify = min_count >= 2 and len(label_counts) > 1
if not can_stratify:
    print("\nStratified split disabled because at least one intent has fewer than 2 samples.")

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2 if len(X) >= 20 else 0.25,
    random_state=42,
    stratify=y if can_stratify else None,
)

print(f"\nTrain samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

Intent sample counts:
  advice: 4
  agriculture: 7
  capabilities: 3
  command: 4
  date_question: 4
  disease_info: 10
  education: 2
  goodbye: 5
  gratitude: 4
  greeting: 18
  help: 2
  identity: 14
  maize_question: 6
  maize_symptom: 7
  monitoring: 10
  progression: 6
  smalltalk: 4
  uncertainty: 1

Stratified split disabled because at least one intent has fewer than 2 samples.

Train samples: 88
Validation samples: 23


## 4. Build a Text Vectorizer

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)

print(f"Vectorized train shape: {X_train_vec.shape}")
print(f"Vectorized validation shape: {X_val_vec.shape}")

Vectorized train shape: (88, 208)
Vectorized validation shape: (23, 208)


## 5. Train a Baseline Classifier

In [28]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
)
classifier.fit(X_train_vec, y_train)

print("Classifier trained.")

Classifier trained.


## 6. Evaluate Model Performance

In [29]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

label_list = sorted(set(y_train + y_val))

y_pred = classifier.predict(X_val_vec)
acc = accuracy_score(y_val, y_pred)

print(f"Validation accuracy: {acc:.4f}")
print("\nClassification report:")
print(classification_report(y_val, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_val, y_pred, labels=label_list))

Validation accuracy: 0.6522

Classification report:
                precision    recall  f1-score   support

        advice       0.50      1.00      0.67         1
  capabilities       0.00      0.00      0.00         0
       command       1.00      1.00      1.00         2
 date_question       0.00      0.00      0.00         1
  disease_info       0.75      1.00      0.86         3
     gratitude       1.00      1.00      1.00         1
      greeting       1.00      1.00      1.00         3
          help       1.00      1.00      1.00         1
      identity       0.67      0.33      0.44         6
maize_question       1.00      1.00      1.00         2
 maize_symptom       0.00      0.00      0.00         0
    monitoring       0.00      0.00      0.00         1
   progression       0.00      0.00      0.00         1
     smalltalk       0.00      0.00      0.00         1

      accuracy                           0.65        23
     macro avg       0.49      0.52      0.50     

## 7. Save the Trained Model as `askinggreetingmodel`

In [35]:
import json
import joblib
from sklearn.pipeline import Pipeline

# Build response map (intent -> responses)
response_map = {}
for intent, group in df_combined.groupby("intent_clean"):
    responses = sorted(group["response_clean"].dropna().unique().tolist())
    response_map[intent] = responses

intent_list = sorted(df_combined["intent_clean"].unique().tolist())

# Save pipeline (vectorizer + classifier) for app.py
text_model = Pipeline([
    ("tfidf", vectorizer),
    ("clf", classifier),
])

# Single final model name requested by user
asking_greeting_model_path = DATA_DIR / "askinggreetingmodel.joblib"
asking_greeting_response_path = DATA_DIR / "askinggreetingmodel_response_map.json"
asking_greeting_labels_path = DATA_DIR / "askinggreetingmodel_labels.json"

joblib.dump(text_model, asking_greeting_model_path)
with asking_greeting_response_path.open("w", encoding="utf-8") as f:
    json.dump(response_map, f, ensure_ascii=False, indent=2)
with asking_greeting_labels_path.open("w", encoding="utf-8") as f:
    json.dump({"intent_list": intent_list}, f, ensure_ascii=False, indent=2)

print("Saved asking/greeting artifacts:")
print(f"- {asking_greeting_model_path}")
print(f"- {asking_greeting_response_path}")
print(f"- {asking_greeting_labels_path}")

Saved asking/greeting artifacts:
- d:\CAPSTONE PROJECT\datasets\maize trainningsystem\askinggreetingmodel.joblib
- d:\CAPSTONE PROJECT\datasets\maize trainningsystem\askinggreetingmodel_response_map.json
- d:\CAPSTONE PROJECT\datasets\maize trainningsystem\askinggreetingmodel_labels.json


## 8. Test Inference with Sample Prompts

In [31]:
import random


def predict_intent(text: str):
    intent = text_model.predict([text])[0]
    proba = text_model.predict_proba([text])[0] if hasattr(text_model, "predict_proba") else None
    confidence = float(max(proba)) if proba is not None else None
    responses = response_map.get(intent, ["I am not sure about that."])
    response = random.choice(responses) if responses else "I am not sure about that."
    return intent, confidence, response


samples = [
    "hello",
    "how do I prevent maize blight",
    "what is gray leaf spot",
    "can you help me",
    "thank you",
]

for sample in samples:
    intent, confidence, response = predict_intent(sample)
    conf_text = f"{confidence:.2f}" if confidence is not None else "N/A"
    print(f"\nInput: {sample}")
    print(f"Intent: {intent} (confidence: {conf_text})")
    print(f"Response: {response}")


Input: hello
Intent: greeting (confidence: 0.10)
Response: Hello! I am Maize Assistant, developed by YUMVAGUSENGA PATRICK. How can I help with your maize crops today?

Input: how do I prevent maize blight
Intent: agriculture (confidence: 0.07)
Response: Use proper fertilizer, watering, and pest control.

Input: what is gray leaf spot
Intent: disease_info (confidence: 0.22)
Response: Rust causes reddish-orange spots on maize leaves.

Input: can you help me
Intent: capabilities (confidence: 0.22)
Response: I detect maize diseases, track crop health, and give farming advice.

Input: thank you
Intent: gratitude (confidence: 0.34)
Response: Happy to help! Remember to monitor your crops regularly for any disease progression.


## Disease Knowledge Base Integration (v1.0)

Connect text-based assistant with image-based disease detection for complete learning support

In [36]:
# Disease Knowledge Base v1.0 - Learning Integration
import json

# Comprehensive disease knowledge for text-based learning
DISEASE_KNOWLEDGE_BASE_V1 = {
    "greeting": {
        "category": "Greeting",
        "responses": [
            "Welcome to the Maize Disease Monitoring AI Assistant!",
            "Hello! I'm here to help you identify and manage maize diseases.",
            "Hi! Ask me about disease identification, stages, or treatment options."
        ]
    },
    
    "healthy_maize": {
        "category": "Plant Health Status",
        "description": "Maize plant with no disease symptoms",
        "stages": ["Healthy - Normal growth"],
        "identification": "Leaves are uniformly green, firm, no spots or streaks",
        "management": "Continue regular monitoring and good agronomic practices",
        "yield_risk": "No yield loss expected"
    },
    
    "common_rust": {
        "category": "Fungal Disease",
        "description": "Orange-brown rust pustules on leaf surface",
        "stages": [
            "Early: Small (1-3mm) orange pustules on lower leaves",
            "Moderate: Pustules grow (5-10mm), spread to mid-leaves",
            "Severe: Large pustules (>10mm), extensive spread"
        ],
        "identification": "Orange pustules arranged in rows, easily rub off, yellow halo around pustules",
        "cause": "Fungal disease spread by windborne spores",
        "favorable_conditions": "Warm (20-25°C) and humid conditions",
        "progression_time": "7-14 days per stage",
        "management": [
            "Apply fungicide every 10-14 days (Propiconazole, Mancozeb)",
            "Remove heavily infected leaves",
            "Improve air circulation"
        ],
        "yield_risk": "10-40% yield loss if untreated"
    },
    
    "gray_leaf_spot": {
        "category": "Fungal Disease",
        "description": "Rectangular gray lesions on leaf blade",
        "stages": [
            "Early: Small (2-3mm) rectangular gray lesions with brown borders",
            "Moderate: Lesions enlarge (5-8mm), spread to mid-leaves",
            "Severe: Large necrotic patches, leaf death, defoliation"
        ],
        "identification": "KEY: Rectangular lesion shape (not circular), gray center with reddish-brown border",
        "cause": "Cercospora zeae-maydis fungus, spreads via rain splash and airborne spores",
        "favorable_conditions": "Warm (20-28°C), humid, wet conditions",
        "progression_time": "10-21 days per stage",
        "management": [
            "Start fungicide early (V6-V8 growth stage)",
            "Apply every 10-14 days",
            "Improve air circulation",
            "Avoid overhead irrigation",
            "Plant spacing for airflow"
        ],
        "yield_risk": "15-50% yield loss if untreated"
    },
    
    "leaf_blight": {
        "category": "Fungal Disease",
        "description": "Large irregular necrotic lesions with tissue death",
        "stages": [
            "Early: Water-soaked, elongated lesions with yellow halo",
            "Moderate: Lesions enlarge, tissue dies (necrosis)",
            "Severe: Extensive dead areas, leaf shredding"
        ],
        "identification": "Large (1-10cm) irregular lesions, initially water-soaked, black tissue in center",
        "cause": "Northern corn leaf blight (Exserohilum turcicum)",
        "favorable_conditions": "Cool (15-20°C), wet conditions",
        "progression_time": "5-10 days per stage",
        "management": [
            "Use resistant hybrids (best option)",
            "Apply fungicide at first sign",
            "Rotate crops",
            "Remove infected crop debris"
        ],
        "yield_risk": "20-60% yield loss if ear leaves affected"
    },
    
    "downy_mildew": {
        "category": "Fungal Disease",
        "description": "Yellow striping with fuzzy white growth on leaf undersurface",
        "stages": [
            "Early: Light yellow streaks, fuzzy whitish growth below",
            "Moderate: Pronounced yellow/white stripes, dense sporulation",
            "Severe: Severe yellowing, plant stunting, leaf death"
        ],
        "identification": "KEY: Yellow striping + WHITE FUZZY GROWTH on leaf underside",
        "cause": "Peronosclerospora sorghi fungus",
        "favorable_conditions": "Cool (18-22°C), wet, humid conditions",
        "progression_time": "3-7 days if seedling infection (rapid)",
        "management": [
            "Use resistant varieties (prevention is key)",
            "Apply metalaxyl-based fungicide early",
            "Maintain proper plant spacing",
            "Reduce leaf wetness"
        ],
        "yield_risk": "30-70% with early infection, 10-20% with late infection"
    },
    
    "maize_streak_virus": {
        "category": "Viral Disease",
        "description": "Yellow streaks running parallel to leaf veins",
        "stages": [
            "Early: Thin yellow streaks on emerging leaves",
            "Moderate: Pronounced yellow/chlorotic streaks, stunting",
            "Severe: Necrotic streaks, severe dwarfing, poor ears"
        ],
        "identification": "KEY: STREAKS parallel to veins (not random), systemic throughout plant",
        "cause": "Maize streak virus (MSV) transmitted by leafhoppers",
        "progression_time": "7-14 days per stage",
        "management": [
            "Control leafhopper vectors with insecticide",
            "Remove infected plants immediately",
            "Plant resistant varieties",
            "Use trap crops"
        ],
        "yield_risk": "30-100% depending on infection timing"
    },
    
    "maize_lethal_necrosis": {
        "category": "Viral Disease - CRITICAL",
        "description": "Rapid necrosis and plant death syndrome",
        "stages": [
            "Early: Chlorotic streaks, initial necrosis",
            "Moderate: Rapid progression, extensive necrosis",
            "Critical: Complete leaf death, plant drying, plant death"
        ],
        "identification": "KEY: RAPID progression (3-5 days), affects young and old leaves, plant drying rapidly",
        "cause": "Combination of viruses (SCMV + MNSV or MMV)",
        "progression_time": "3-5 days from Stage 1 to Stage 2, then 2-3 days to death",
        "management": [
            "IMMEDIATE: Isolate and remove infected plants",
            "Control insect vectors",
            "Monitor neighboring crops closely",
            "Use resistant varieties if available",
            "Consult extension officer"
        ],
        "yield_risk": "Catastrophic: 70-100% yield loss"
    },
    
    "asking": {
        "category": "General Questions",
        "responses": [
            "I can help you with: disease identification, disease stages, management strategies, and treatment recommendations.",
            "Ask me about any maize disease and I'll provide detailed information on identification, progression, and management."
        ]
    }
}

# Enhanced prediction with disease context
def predict_intent_with_context(text: str):
    """Predict intent and provide disease knowledge context"""
    intent, confidence, response = predict_intent(text)
    
    # Get disease knowledge if available
    disease_info = DISEASE_KNOWLEDGE_BASE_V1.get(intent, {})
    
    result = {
        "input_text": text,
        "predicted_intent": intent,
        "confidence": confidence,
        "response": response,
        "has_disease_info": bool(disease_info),
        "disease_info": disease_info
    }
    
    return result

# Test with disease-related queries
print("="*80)
print("DISEASE KNOWLEDGE BASE - TEXT ASSISTANT v1.0")
print("="*80)

test_queries = [
    "hello",
    "what is gray leaf spot",
    "how do I identify common rust",
    "what should I do for maize streak virus",
    "tell me about leaf blight",
    "what is maize lethal necrosis",
]

for query in test_queries:
    result = predict_intent_with_context(query)
    print(f"\nQuery: '{query}'")
    print(f"   Intent: {result['predicted_intent']} (confidence: {result['confidence']:.2f})")
    print(f"   Response: {result['response']}")
    
    if result['has_disease_info'] and 'description' in result['disease_info']:
        info = result['disease_info']
        print(f"   Disease: {info.get('description', 'N/A')}")
        print(f"   Risk: {info.get('yield_risk', 'N/A')}")

print("\n" + "="*80)
print("Disease Knowledge Base Integrated Successfully!")
print("="*80)

DISEASE KNOWLEDGE BASE - TEXT ASSISTANT v1.0

Query: 'hello'
   Intent: greeting (confidence: 0.10)
   Response: Good morning! I can help you detect maize diseases.

Query: 'what is gray leaf spot'
   Intent: disease_info (confidence: 0.22)
   Response: Blight is a fungal disease causing leaf lesions.

Query: 'how do I identify common rust'
   Intent: progression (confidence: 0.08)
   Response: Tracking disease progression lets you know if treatment is working or if the disease is worsening. It moves beyond simple detection to help you make informed, timely farming decisions.

Query: 'what should I do for maize streak virus'
   Intent: agriculture (confidence: 0.11)
   Response: Depends on soil type; soil testing is recommended.

Query: 'tell me about leaf blight'
   Intent: disease_info (confidence: 0.08)
   Response: Gray Leaf Spot is a fungal disease affecting leaves.

Query: 'what is maize lethal necrosis'
   Intent: maize_question (confidence: 0.12)
   Response: Upload an image an

## Save `askinggreetingmodel` (single model export)

In [37]:
# Save Text Model with final single-name export
import json
import joblib
from datetime import datetime

TEXT_MODEL_VERSION = "1.0"
EXPORT_TIMESTAMP = datetime.now().isoformat()

# Single model filename requested by user
asking_greeting_model_path = DATA_DIR / "askinggreetingmodel.joblib"
asking_greeting_response_path = DATA_DIR / "askinggreetingmodel_response_map.json"
asking_greeting_labels_path = DATA_DIR / "askinggreetingmodel_labels.json"
asking_greeting_knowledge_path = DATA_DIR / "askinggreetingmodel_knowledge_base.json"

text_model_metadata = {
    "model_name": "askinggreetingmodel",
    "version": TEXT_MODEL_VERSION,
    "timestamp": EXPORT_TIMESTAMP,
    "model_type": "Logistic Regression (TF-IDF)",
    "intents": intent_list,
    "num_training_samples": len(df_combined),
    "features": [
        "Intent classification",
        "Disease question answering",
        "Learning support",
        "Farmer-friendly guidance"
    ]
}

print(f"Exporting Text Model as askinggreetingmodel v{TEXT_MODEL_VERSION}...")

# Save exactly one model file for this notebook
joblib.dump(text_model, str(asking_greeting_model_path))
print(f"✓ Model saved: {asking_greeting_model_path.name}")

with open(str(asking_greeting_response_path), 'w', encoding='utf-8') as f:
    json.dump(response_map, f, ensure_ascii=False, indent=2)
print(f"✓ Response map saved: {asking_greeting_response_path.name}")

with open(str(asking_greeting_labels_path), 'w', encoding='utf-8') as f:
    json.dump({"intent_list": intent_list, "metadata": text_model_metadata}, f, ensure_ascii=False, indent=2)
print(f"✓ Labels saved: {asking_greeting_labels_path.name}")

with open(str(asking_greeting_knowledge_path), 'w', encoding='utf-8') as f:
    json.dump(DISEASE_KNOWLEDGE_BASE_V1, f, ensure_ascii=False, indent=2)
print(f"✓ Knowledge base saved: {asking_greeting_knowledge_path.name}")

print("\n" + "="*80)
print("ASKING/GREETING MODEL SUMMARY:")
print("="*80)
print("Model Name:         askinggreetingmodel")
print(f"Version:            {TEXT_MODEL_VERSION}")
print(f"Export Date:        {EXPORT_TIMESTAMP}")
print("Primary Files:")
print(f"  - {asking_greeting_model_path.name}")
print(f"  - {asking_greeting_response_path.name}")
print(f"  - {asking_greeting_labels_path.name}")
print(f"  - {asking_greeting_knowledge_path.name}")
print(f"\nIntents Supported:  {', '.join(intent_list)}")
print(f"Training Samples:   {len(df_combined)}")
print("Model Type:         Logistic Regression + TF-IDF Vectorizer")
print("Use in app.py:      Load 'askinggreetingmodel.joblib'")
print("="*80)

Exporting Text Model as askinggreetingmodel v1.0...
✓ Model saved: askinggreetingmodel.joblib
✓ Response map saved: askinggreetingmodel_response_map.json
✓ Labels saved: askinggreetingmodel_labels.json
✓ Knowledge base saved: askinggreetingmodel_knowledge_base.json

ASKING/GREETING MODEL SUMMARY:
Model Name:         askinggreetingmodel
Version:            1.0
Export Date:        2026-05-13T13:18:06.929872
Primary Files:
  - askinggreetingmodel.joblib
  - askinggreetingmodel_response_map.json
  - askinggreetingmodel_labels.json
  - askinggreetingmodel_knowledge_base.json

Intents Supported:  advice, agriculture, capabilities, command, date_question, disease_info, education, goodbye, gratitude, greeting, help, identity, maize_question, maize_symptom, monitoring, progression, smalltalk, uncertainty
Training Samples:   111
Model Type:         Logistic Regression + TF-IDF Vectorizer
Use in app.py:      Load 'askinggreetingmodel.joblib'
